# 03 Results tables and figures
Reads `report.json` and per-image scores from a results folder and makes the tables/figures used in slides and the thesis. Label synthetic results as synthetic.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import roc_curve

RES = Path('results/smoke')
rep = json.loads((RES / 'report.json').read_text())
print('golden bit-exact:', rep['golden_bit_exact'], '| optimized kernels max LSB diff:', rep['optimized_kernel_max_lsb_diff'])
print('INT8 macro-F1 drop (pp):', round(rep['int8_macro_f1_drop_pp'], 2))
pd.DataFrame({s: {m: v['macro_f1'] for m, v in rep['tile'][s].items()} for s in rep['tile']})

In [ ]:
# farm-level table with bootstrap CIs (threshold chosen on validation only)
rows = []
for s in ('test', 'test_shift'):
    f = rep['farm_int8'].get(s)
    if f: rows.append(dict(split=s, n_images=f['n_images'], sens=f['sensitivity'], sens_ci=f['sensitivity_ci'][1:], spec=f['specificity'], spec_ci=f['specificity_ci'][1:], auc=f['auc']))
pd.DataFrame(rows)

In [ ]:
# ROC of the farm risk score
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for s in ('test', 'test_shift'):
    d = pd.read_csv(RES / f'farm_scores_{s}.csv')
    if d.y_exw.nunique() == 2:
        fpr, tpr, _ = roc_curve(d.y_exw, d.risk); ax.plot(fpr, tpr, label=s)
ax.plot([0, 1], [0, 1], 'k:'); ax.set_xlabel('false alarm rate'); ax.set_ylabel('sensitivity'); ax.legend()
fig.savefig('docs/figures/farm_roc.png', dpi=150, bbox_inches='tight')

In [ ]:
# latency table (lab CPU only: phone and DE1-SoC numbers come from their own benchmarks)
pd.DataFrame(rep['benchmark_lab_cpu'])